In [1]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 136,
   "id": "517b8e95",
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "from sklearn.cluster import KMeans\n",
    "from sklearn.cluster import AgglomerativeClustering\n",
    "\n",
    "from sklearn.metrics import (\n",
    "    silhouette_score,\n",
    "    davies_bouldin_score\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 137,
   "id": "395722c8",
   "metadata": {},
   "outputs": [],
   "source": [
    "CSV_PATH = \"dataset/raw_wholesale_customers.csv\"\n",
    "\n",
    "OUT_PATH = \"dataset/segmented_wholesale_customers.csv\"\n",
    "\n",
    "FEATURES = [\n",
    "    \"Fresh\",\n",
    "    \"Milk\",\n",
    "    \"Grocery\",\n",
    "    \"Frozen\",\n",
    "    \"Detergents_Paper\",\n",
    "    \"Delicassen\"\n",
    "]\n",
    "\n",
    "RANDOM_STATE = 42\n",
    "\n",
    "K = 5"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 138,
   "id": "5e856abc",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== DATASET HEAD ==========\n",
      "   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen\n",
      "0        2       3  12669  9656     7561     214              2674        1338\n",
      "1        2       3   7057  9810     9568    1762              3293        1776\n",
      "2        2       3   6353  8808     7684    2405              3516        7844\n",
      "3        1       3  13265  1196     4221    6404               507        1788\n",
      "4        2       3  22615  5410     7198    3915              1777        5185\n",
      "\n",
      "========== DATASET INFO ==========\n",
      "<class 'pandas.DataFrame'>\n",
      "RangeIndex: 440 entries, 0 to 439\n",
      "Data columns (total 8 columns):\n",
      " #   Column            Non-Null Count  Dtype\n",
      "---  ------            --------------  -----\n",
      " 0   Channel           440 non-null    int64\n",
      " 1   Region            440 non-null    int64\n",
      " 2   Fresh             440 non-null    int64\n",
      " 3   Milk              440 non-null    int64\n",
      " 4   Grocery           440 non-null    int64\n",
      " 5   Frozen            440 non-null    int64\n",
      " 6   Detergents_Paper  440 non-null    int64\n",
      " 7   Delicassen        440 non-null    int64\n",
      "dtypes: int64(8)\n",
      "memory usage: 27.6 KB\n",
      "None\n"
     ]
    }
   ],
   "source": [
    "df = pd.read_csv(CSV_PATH)\n",
    "\n",
    "print(\"========== DATASET HEAD ==========\")\n",
    "print(df.head())\n",
    "\n",
    "print(\"\\n========== DATASET INFO ==========\")\n",
    "print(df.info())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 139,
   "id": "437e90a2",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Selected Features\n",
      "   Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen\n",
      "0  12669  9656     7561     214              2674        1338\n",
      "1   7057  9810     9568    1762              3293        1776\n",
      "2   6353  8808     7684    2405              3516        7844\n",
      "3  13265  1196     4221    6404               507        1788\n",
      "4  22615  5410     7198    3915              1777        5185\n"
     ]
    }
   ],
   "source": [
    "X = df[FEATURES].copy()\n",
    "\n",
    "print(\"Selected Features\")\n",
    "\n",
    "print(X.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 140,
   "id": "126c31d7",
   "metadata": {},
   "outputs": [],
   "source": [
    "def iqr_limits(series, k=1.5):\n",
    "\n",
    "    q1 = series.quantile(0.25)\n",
    "\n",
    "    q3 = series.quantile(0.75)\n",
    "\n",
    "    iqr = q3 - q1\n",
    "\n",
    "    lower = q1 - k * iqr\n",
    "\n",
    "    upper = q3 + k * iqr\n",
    "\n",
    "    return lower, upper"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 141,
   "id": "1c2f65d6",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Fresh                Lower=-17581.25 Upper=37642.75\n",
      "Milk                 Lower=-6952.88 Upper=15676.12\n",
      "Grocery              Lower=-10601.12 Upper=23409.88\n",
      "Frozen               Lower=-3475.75 Upper=7772.25\n",
      "Detergents_Paper     Lower=-5241.12 Upper=9419.88\n",
      "Delicassen           Lower=-1709.75 Upper=3938.25\n",
      "     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen\n",
      "0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00\n",
      "1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00\n",
      "2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25\n",
      "3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00\n",
      "4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25\n"
     ]
    }
   ],
   "source": [
    "for column in FEATURES:\n",
    "\n",
    "    low, high = iqr_limits(X[column])\n",
    "\n",
    "    X[column] = X[column].clip(\n",
    "        lower=low,\n",
    "        upper=high\n",
    "    )\n",
    "\n",
    "    print(\n",
    "        f\"{column:20} \"\n",
    "        f\"Lower={low:.2f} \"\n",
    "        f\"Upper={high:.2f}\"\n",
    "    )\n",
    "\n",
    "    df[FEATURES] = X\n",
    "\n",
    "print(X.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 142,
   "id": "a4db2ae8",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Scaled Shape\n",
      "(440, 6)\n"
     ]
    }
   ],
   "source": [
    "scaler = StandardScaler()\n",
    "\n",
    "X_scaled = scaler.fit_transform(X)\n",
    "\n",
    "print(\"Scaled Shape\")\n",
    "\n",
    "print(X_scaled.shape)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 143,
   "id": "1ae8e9c7",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== ELBOW METHOD ==========\n",
      "k = 1 | SSE = 2640.00\n",
      "k = 2 | SSE = 1728.19\n",
      "k = 3 | SSE = 1363.45\n",
      "k = 4 | SSE = 1202.41\n",
      "k = 5 | SSE = 1070.15\n",
      "k = 6 | SSE = 964.76\n",
      "k = 7 | SSE = 921.14\n",
      "k = 8 | SSE = 776.63\n",
      "k = 9 | SSE = 726.88\n",
      "k = 10 | SSE = 707.41\n"
     ]
    }
   ],
   "source": [
    "print(\"========== ELBOW METHOD ==========\")\n",
    "\n",
    "sse = []\n",
    "\n",
    "for k in range(1, 11):\n",
    "\n",
    "    model = KMeans(\n",
    "        n_clusters=k,\n",
    "        random_state=RANDOM_STATE,\n",
    "        n_init=\"auto\"\n",
    "    )\n",
    "\n",
    "    model.fit(X_scaled)\n",
    "\n",
    "    sse.append(model.inertia_)\n",
    "\n",
    "    print(f\"k = {k} | SSE = {model.inertia_:.2f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 144,
   "id": "d8953d31",
   "metadata": {},
   "outputs": [
    {
     "data": {
     
      "text/plain": [
       "<Figure size 700x500 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "plt.figure(figsize=(7,5))\n",
    "\n",
    "plt.plot(range(1,11), sse, marker=\"o\")\n",
    "\n",
    "plt.title(\"Elbow Method\")\n",
    "\n",
    "plt.xlabel(\"Number of Clusters (k)\")\n",
    "\n",
    "plt.ylabel(\"SSE (Inertia)\")\n",
    "\n",
    "plt.grid(True)\n",
    "\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 145,
   "id": "476b2d11",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== SAMPLE WITH KMEANS CLUSTERS ==========\n",
      "   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \\\n",
      "0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   \n",
      "1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   \n",
      "2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   \n",
      "3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   \n",
      "4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   \n",
      "\n",
      "   Delicassen  Cluster  \n",
      "0     1338.00        0  \n",
      "1     1776.00        0  \n",
      "2     3938.25        0  \n",
      "3     1788.00        3  \n",
      "4     3938.25        3  \n"
     ]
    }
   ],
   "source": [
    "kmeans = KMeans(\n",
    "    n_clusters=K,\n",
    "    random_state=RANDOM_STATE,\n",
    "    n_init=\"auto\"\n",
    ")\n",
    "\n",
    "kmeans_labels = kmeans.fit_predict(X_scaled)\n",
    "\n",
    "df[\"Cluster\"] = kmeans_labels\n",
    "\n",
    "print(\"========== SAMPLE WITH KMEANS CLUSTERS ==========\")\n",
    "\n",
    "print(df.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 146,
   "id": "0f88a9ce",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== KMEANS METRICS ==========\n",
      "Silhouette Score : 0.283\n",
      "Davies-Bouldin Index : 1.270\n"
     ]
    }
   ],
   "source": [
    "kmeans_silhouette = silhouette_score(\n",
    "    X_scaled,\n",
    "    kmeans_labels\n",
    ")\n",
    "\n",
    "kmeans_dbi = davies_bouldin_score(\n",
    "    X_scaled,\n",
    "    kmeans_labels\n",
    ")\n",
    "\n",
    "print(\"========== KMEANS METRICS ==========\")\n",
    "\n",
    "print(f\"Silhouette Score : {kmeans_silhouette:.3f}\")\n",
    "\n",
    "print(f\"Davies-Bouldin Index : {kmeans_dbi:.3f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 147,
   "id": "72e4ad2a",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== CLUSTER CENTERS ==========\n",
      "            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen\n",
      "Cluster                                                                     \n",
      "0         9202.67   6833.30   9104.12  1326.16           3280.12     1871.76\n",
      "1         8376.23   2150.65   3160.63  1646.33            779.25      674.02\n",
      "2        17461.54  13805.60  17524.12  4120.57           5460.56     3583.64\n",
      "3        22346.70   3409.14   3969.33  5819.60            583.07     1566.95\n",
      "4         4916.98  10768.85  18350.13  1212.37           7780.02      981.37\n"
     ]
    }
   ],
   "source": [
    "centers = scaler.inverse_transform(\n",
    "    kmeans.cluster_centers_\n",
    ")\n",
    "\n",
    "centers_df = pd.DataFrame(\n",
    "    centers,\n",
    "    columns=FEATURES\n",
    ")\n",
    "\n",
    "centers_df.index.name = \"Cluster\"\n",
    "\n",
    "print(\"========== CLUSTER CENTERS ==========\")\n",
    "\n",
    "print(centers_df.round(2))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7f6cd5c5",
   "metadata": {},
   "source": [
    "## Second Clustering Algorithm – Agglomerative Clustering\n",
    "\n",
    "For the second clustering method, I selected **Agglomerative Clustering**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 148,
   "id": "5c2b9aca",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== AGGLOMERATIVE SAMPLE ==========\n",
      "   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \\\n",
      "0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   \n",
      "1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   \n",
      "2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   \n",
      "3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   \n",
      "4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   \n",
      "\n",
      "   Delicassen  Cluster  Agglomerative_Cluster  \n",
      "0     1338.00        0                      4  \n",
      "1     1776.00        0                      4  \n",
      "2     3938.25        0                      0  \n",
      "3     1788.00        3                      3  \n",
      "4     3938.25        3                      0  \n"
     ]
    }
   ],
   "source": [
    "agglomerative = AgglomerativeClustering(\n",
    "    n_clusters=5\n",
    ")\n",
    "\n",
    "agg_labels = agglomerative.fit_predict(X_scaled)\n",
    "\n",
    "df[\"Agglomerative_Cluster\"] = agg_labels\n",
    "\n",
    "print(\"========== AGGLOMERATIVE SAMPLE ==========\")\n",
    "\n",
    "print(df.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 149,
   "id": "aca4c9a0",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== AGGLOMERATIVE METRIC ==========\n",
      "Silhouette Score : 0.218\n"
     ]
    }
   ],
   "source": [
    "agg_silhouette = silhouette_score(\n",
    "    X_scaled,\n",
    "    agg_labels\n",
    ")\n",
    "\n",
    "print(\"========== AGGLOMERATIVE METRIC ==========\")\n",
    "\n",
    "print(f\"Silhouette Score : {agg_silhouette:.3f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 150,
   "id": "456b24fa",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== COMPARISON ==========\n",
      "K-Means Silhouette Score        : 0.283\n",
      "Agglomerative Silhouette Score : 0.218\n",
      "\n",
      "K-Means produced better separated clusters.\n"
     ]
    }
   ],
   "source": [
    "print(\"========== COMPARISON ==========\")\n",
    "\n",
    "print(f\"K-Means Silhouette Score        : {kmeans_silhouette:.3f}\")\n",
    "\n",
    "print(f\"Agglomerative Silhouette Score : {agg_silhouette:.3f}\")\n",
    "\n",
    "if kmeans_silhouette > agg_silhouette:\n",
    "    print(\"\\nK-Means produced better separated clusters.\")\n",
    "elif agg_silhouette > kmeans_silhouette:\n",
    "    print(\"\\nAgglomerative Clustering produced better separated clusters.\")\n",
    "else:\n",
    "    print(\"\\nBoth algorithms produced similar cluster separation.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 151,
   "id": "1c15bc0b",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "========== SANITY CHECK ==========\n",
      "   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \\\n",
      "0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   \n",
      "1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   \n",
      "2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   \n",
      "\n",
      "   Delicassen  Cluster  Agglomerative_Cluster  \n",
      "0     1338.00        0                      4  \n",
      "1     1776.00        0                      4  \n",
      "2     3938.25        0                      0  \n"
     ]
    }
   ],
   "source": [
    "sample_clients = df.loc[\n",
    "    [0, 1, 2],\n",
    "    [\n",
    "        \"Channel\",\n",
    "        \"Region\",\n",
    "        \"Fresh\",\n",
    "        \"Milk\",\n",
    "        \"Grocery\",\n",
    "        \"Frozen\",\n",
    "        \"Detergents_Paper\",\n",
    "        \"Delicassen\",\n",
    "        \"Cluster\",\n",
    "        \"Agglomerative_Cluster\"\n",
    "    ]\n",
    "]\n",
    "\n",
    "print(\"========== SANITY CHECK ==========\")\n",
    "\n",
    "print(sample_clients)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 152,
   "id": "cede9945",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Segmented dataset saved successfully!\n",
      "Location: dataset/segmented_wholesale_customers.csv\n"
     ]
    }
   ],
   "source": [
    "df.to_csv(\n",
    "    OUT_PATH,\n",
    "    index=False\n",
    ")\n",
    "\n",
    "print(\"Segmented dataset saved successfully!\")\n",
    "\n",
    "print(f\"Location: {OUT_PATH}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.14.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

{'cells': [{'cell_type': 'code',
   'execution_count': 136,
   'id': '517b8e95',
   'metadata': {},
   'outputs': [],
   'source': ['import pandas as pd\n',
    'import matplotlib.pyplot as plt\n',
    '\n',
    'from sklearn.preprocessing import StandardScaler\n',
    'from sklearn.cluster import KMeans\n',
    'from sklearn.cluster import AgglomerativeClustering\n',
    '\n',
    'from sklearn.metrics import (\n',
    '    silhouette_score,\n',
    '    davies_bouldin_score\n',
    ')']},
  {'cell_type': 'code',
   'execution_count': 137,
   'id': '395722c8',
   'metadata': {},
   'outputs': [],
   'source': ['CSV_PATH = "dataset/raw_wholesale_customers.csv"\n',
    '\n',
    'OUT_PATH = "dataset/segmented_wholesale_customers.csv"\n',
    '\n',
    'FEATURES = [\n',
    '    "Fresh",\n',
    '    "Milk",\n',
    '    "Grocery",\n',
    '    "Frozen",\n',
    '    "Detergents_Paper",\n',
    '    "Delicassen"\n',
    ']\n',
    '\n',
    'RANDOM_STATE = 42\n',
    '\n',
    'K = 5']},